<a href="https://colab.research.google.com/github/granthbharill-sudo/AI-Agents/blob/main/Task_01_Resume_CoverLetter_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers accelerate torch
!pip install -q gradio

import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. Setup Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# 2. Initialize Models (Using the efficient 1.5B models from your original framework)
deepseek_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
deepseek_tokenizer = AutoTokenizer.from_pretrained(deepseek_name)
deepseek_model = AutoModelForCausalLM.from_pretrained(
    deepseek_name, torch_dtype=torch.float16
).to(device)

qwen_name = "Qwen/Qwen2.5-1.5B-Instruct"
qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_name)
qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_name, torch_dtype=torch.float16
).to(device)


# 3. Agent 1: DeepSeek ATS Optimizer & Content Generator
def deepseek_resume_generation(user_experience, job_description):
    prompt = f"""
You are an expert Resume Writer and ATS (Applicant Tracking System) Optimizer.
Analyze the user's background and the target job description to build a highly tailored resume summary and experience bullet points. Ensure high keyword matching for ATS tracking.

User Experience/Background:
{user_experience}

Target Job Description:
{job_description}

Output a professional, ATS-optimized Resume section. Include clear headers and action-oriented bullet points.
"""
    inputs = deepseek_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(device)
    output = deepseek_model.generate(**inputs, max_new_tokens=600, temperature=0.5, do_sample=True)
    return deepseek_tokenizer.decode(output[0], skip_special_tokens=True)


# 4. Agent 2: Qwen Personalized Cover Letter Writer
def qwen_cover_letter_generation(user_experience, job_description, custom_prompt):
    prompt = f"""
You are a Professional Career Coach. Write a personalized, persuasive cover letter based on the user's details, the target role, and their specific writing tone request.

User Experience:
{user_experience}

Target Job:
{job_description}

Custom Instructions/Tone:
{custom_prompt}

Output a beautifully structured cover letter ready for submission. Do not include placeholders like '[Insert Date]'.
"""
    inputs = qwen_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(device)
    output = qwen_model.generate(**inputs, max_new_tokens=600, temperature=0.7, do_sample=True)
    return qwen_tokenizer.decode(output[0], skip_special_tokens=True)


# 5. Main Workflow Orchestrator
def run_generator(user_experience, job_description, custom_prompt):
    # Generate tailored resume components
    resume_output = deepseek_resume_generation(user_experience, job_description)

    # Generate the custom cover letter
    cover_letter_output = qwen_cover_letter_generation(user_experience, job_description, custom_prompt)

    # Combine outputs cleanly into a single final view
    final_report = f"""==================================================
1. TAILORED & ATS-OPTIMIZED RESUME CONTENT
==================================================
{resume_output}

==================================================
2. PERSONALIZED COVER LETTER
==================================================
{cover_letter_output}
"""
    return final_report


# 6. Gradio Interface Layout
with gr.Blocks() as demo:
    gr.Markdown("# 🚀 AI Resume & Cover Letter Generator (ATS Optimized)")
    gr.Markdown("Provide your background details and the target job description to generate optimized, tailored recruitment materials.")

    with gr.Row():
        with gr.Column():
            user_experience = gr.Textbox(
                label="Your Professional Background / Paste Existing Resume",
                lines=6,
                placeholder="List your jobs, skills, and past achievements here..."
            )
            job_description = gr.Textbox(
                label="Target Job Description / Role Details",
                lines=6,
                placeholder="Paste the requirements and description of the role you are targeting..."
            )
            custom_prompt = gr.Textbox(
                label="Cover Letter Custom Instructions (Optional)",
                lines=2,
                value="Make the tone confident, professional, and enthusiastic."
            )
            btn = gr.Button("Generate Application Package", variant="primary")

        with gr.Column():
            answer = gr.Textbox(
                label="Generated Resume & Cover Letter Output",
                lines=18
            )

    btn.click(
        fn=run_generator,
        inputs=[user_experience, job_description, custom_prompt],
        outputs=answer
    )

demo.launch(share=True, debug=True)

Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3de21aa5c9a4693a3d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
